# Cache

Inspect, copy, reuse, and manage results saved by the other notebooks.

Run the setup cell first. Each numbered section has editable inputs followed by a
calculation cell. Run that section again after changing its inputs, or use
**Restart Kernel and Run All Cells** to run the examples from a clean state.
Run each workflow's calculation before its result-detail cells. Every input has
a comment describing its format or choices. Optional fields accept `None`,
`"None"`, or blank text. Rerun setup after updating these files.

Matrices accept `1 2; 3 4`, rows on separate lines, fractions such as `1/3`,
nested lists, and NumPy arrays. Rows usually represent samples and columns features.
Results shown with `cache=cache` are saved automatically for use in any notebook
through `cache.load("name")`. The [Cache notebook](09_cache.ipynb) manages them.

[Setup and notebook index](README.md)

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in this folder, the repository root, or a subfolder.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "CommonUtils.py").is_file() and (p / "notebooks" / "notebook_tools.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the ee2211-solver repository or its notebooks folder.")
for folder in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import Markdown, display
import importlib
import notebook_tools
importlib.reload(notebook_tools)
from notebook_tools import (NotebookCache, matrix, optional_matrix, is_blank,
    number, integer, boolean, variable_names, coordinates, choice, show, show_metrics)

pd.set_option("display.precision", 10)
cache = NotebookCache()
print(f"Python {sys.version.split()[0]} | shared cache: {cache.path}")

## 1. List saved results
Every notebook uses `notebooks/.cache/matrices.sqlite3`. Results survive kernel
restarts and are shared between notebook kernels. The table is newest first.
Rerun this cell to refresh results created in another notebook. Times are UTC.

This cache has its own history, separate from the website. By default it retains
50 entries. Repeating a calculation creates new entries. For a repeated name,
`cache.load(name)` returns the newest saved value. Use an ID to choose an exact
older entry. SQLite is part of Python; no database package is needed.

In [ ]:
entries = cache.table()
print(f"{len(entries)} of {cache.limit} allowed entries")
display(entries)
if entries.empty:
    print("Run any calculation notebook to save its results here.")

## 2. Inspect and copy a result
Set `selection` to a name or ID from the table, or keep `None` to preview the most
recent entry. The plain text is a nested Python list you can paste into an input
cell. Values retain their stored precision. Text class labels are supported.

In another notebook, after its setup cell, use for example
`X = cache.load("prepared_data")` and pass X directly to a calculator, or assign
`x_raw = cache.load("prepared_data")` in its input cell.

In [ ]:
selection = None  # Saved name/ID; None previews the newest entry.

In [ ]:
latest = cache.table()
selected = selection if not is_blank(selection) else (latest.iloc[0]["id"] if not latest.empty else None)
if selected is not None:
    saved_matrix = cache.load(selected)
    show("saved_matrix", saved_matrix)
    display(Markdown("**Copyable Python list**"))
    print(cache.text(selected))
else:
    print("The cache is empty. Run a calculation first.")

## 3. Save a matrix manually
You can save any numeric matrix or class-label array. Set `save_manual = True`
to run this example. Saving repeatedly makes separate history entries.

In [ ]:
save_manual = False  # True saves the matrix below; False skips.
manual_name = 'my_matrix'  # Any descriptive saved-result name.
manual_value = [[1, 2], [3, 4]]  # Numeric matrix or text-label column; nested lists/arrays.

In [ ]:
if boolean(save_manual):
    manual_id = cache.save(manual_name, manual_value, source="Manual notebook input", kind="input")
    print("Saved entry:", manual_id)

## 4. Retention limit, removal, and clearing
These actions are disabled in the example inputs. Set `new_limit` to an integer
from 1 to 1000 to change retention (older excess entries are removed immediately).
Set `remove_id` to delete one entry by its ID. Set `clear_cache = True` to delete
all cached entries. After use, reset the inputs before running the whole notebook
again. These actions affect the shared notebook cache.

In [ ]:
new_limit = None  # Integer 1-1000, or None to leave retention unchanged.
remove_id = None  # Exact entry ID to delete, or None to keep entries.
clear_cache = False  # True deletes all notebook results; False keeps them.

In [ ]:
if not is_blank(new_limit):
    cache.set_limit(integer(new_limit))
if not is_blank(remove_id):
    print("Entries removed:", cache.remove(remove_id))
if boolean(clear_cache):
    cache.clear()
    print("Cache cleared.")
print(f"{len(cache.table())} of {cache.limit} allowed entries")
display(cache.table())